# GraphRAG: answering battery questions from the knowledge graph

1. The LLM turns the question into helper-function calls or a SPARQL query.
2. The query runs against the knowledge graph (only the three helper functions and read-only SPARQL are executed).
3. The LLM phrases the answer from the query result only.

**API keys.** Copy `.env.example` to `.env` and fill in the key for the provider you use. `.env` is git-ignored; never commit keys. Local models run through [Ollama](https://ollama.com) without a key.

The recorded evaluation (300 questions, `evaluation/`) is not re-run here. Section 3 recreates its figures from the recorded results.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # not needed after `pip install -e .`

from batterygraph import BatteryGraph, GraphRAG

# Model names: "claude-sonnet-5", "gpt-5.4", "gemini-3.5-flash", or a local Ollama model such as "gemma4:31b"
MODEL = "gemini-3.5-flash"

bg = BatteryGraph(ROOT / "BatteryTypeJson")
rag = GraphRAG(bg, model=MODEL, verbose=True)

## 1. Ask a question

In [ ]:
rag.answer("Which battery has a higher rated capacity: the Saft VL 5U or the Lishen LR2170SA?")

In [ ]:
# Same question without the knowledge graph (LLM-only baseline)
rag.answer_without_rag("Which battery has a higher rated capacity: the Saft VL 5U or the Lishen LR2170SA?")

## 2. Evaluate on a few questions

The full run is a script: `python scripts/run_eval.py --model <model>` (add `--no-rag` for the baseline).

In [ ]:
import pandas as pd
from batterygraph.evaluate import run_evaluation, summarize

questions = pd.read_csv(ROOT / "evaluation" / "qa_dataset.csv").head(5)
rag.verbose = False
sample = run_evaluation(questions, rag)
summarize(sample)

## 3. Recorded results and figures

Recorded with the question set in `evaluation/qa_dataset.csv`; graded by the answering model; the "identifiably wrong" check by `gemini-3.5-flash`.

* **accuracy** = correct answers / all questions
* **hallucination robustness** = answers that admit missing data ("I don't know") / incorrect answers

In [ ]:
from batterygraph.evaluate import plot_figures

results = pd.read_csv(ROOT / "evaluation" / "results" / "graphrag_evaluation_results.csv")
display(summarize(results).round(3))
plot_figures(results, ROOT / "evaluation" / "figures")

In [ ]:
from IPython.display import SVG, display

for name in ("RAG_Accuracy", "RAG_HallucinationRobustness"):
    display(SVG(filename=str(ROOT / "evaluation" / "figures" / f"{name}.svg")))